# General Explore SHEDS Data (2025)

Converted from R Markdown to Python Jupyter Notebook

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import custom utilities
from utils import read_clean_sheds, get_data_summary

sns.set_palette("husl")

# Read SPSS file - keeping metadata for labels
data_path_file = "/path/to/the/files/SHEDS2025.sav"
sheds, meta = read_clean_sheds(data_path_file)

# Filter out screened respondents
sheds = sheds[sheds['screen'] != 3].copy()

print(f"Shape: {sheds.shape}")
sheds.head()

## Helper Function to Apply SPSS Labels

In [ ]:
def apply_spss_labels(data: pd.DataFrame, variable_name: str, meta) -> pd.Series:
    """
    Apply SPSS value labels to a variable.
    
    Parameters
    ----------
    data : pd.DataFrame
        The dataframe containing the variable
    variable_name : str
        Name of the variable to label
    meta : pyreadstat metadata
        Metadata object from pyreadstat.read_sav()
        
    Returns
    -------
    pd.Series
        Series with labels applied
    """
    if variable_name not in meta.variable_value_labels:
        return data[variable_name].astype(str)
    
    # Get the label mapping {code: label}
    labels = meta.variable_value_labels[variable_name]
    
    # Map values to labels
    return data[variable_name].map(labels).fillna(data[variable_name].astype(str))


def get_variable_label(variable_name: str, meta) -> str:
    """
    Get the variable label (description) from SPSS metadata.
    """
    if variable_name in meta.column_names_to_labels:
        return meta.column_names_to_labels[variable_name]
    return variable_name

## Basic Metrics

In [ ]:
# Basic overview
print("Data Info:")
print(sheds.info())
print("\n" + "="*50 + "\n")

# Summary statistics
print("Summary Statistics:")
print(sheds.describe())
print("\n" + "="*50 + "\n")

# Check completion rate
print("Finished distribution:")
print(sheds['finished'].value_counts())
print("\nFinished proportions:")
print(sheds['finished'].value_counts(normalize=True))

# Total statistics
total_respondents = len(sheds)
total_finished = (sheds['finished'] == 1).sum()
completion_rate = total_finished / total_respondents * 100

print(f"\nTotal respondents: {total_respondents}")
print(f"Finished responses: {total_finished}")
print(f"Completion rate: {completion_rate:.1f}%")

## Age Distribution

In [ ]:
# Age distribution using SPSS labels
finished_data = sheds[sheds['finished'] == 1].copy()
finished_data['agegr_label'] = apply_spss_labels(finished_data, 'agegr', meta)

fig, ax = plt.subplots(figsize=(10, 6))
age_counts = finished_data['agegr_label'].value_counts().sort_index()
age_counts.plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Age Distribution of Respondents', fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Gender Distribution

In [ ]:
# Gender distribution using SPSS labels
gender_data = finished_data.copy()
gender_data['sex_label'] = apply_spss_labels(gender_data, 'sex', meta)

gender_counts = gender_data['sex_label'].value_counts()
gender_pct = gender_counts / gender_counts.sum() * 100

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(gender_pct.index, gender_pct.values, color='coral')

# Add percentage labels
for bar, pct in zip(bars, gender_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{pct:.1f}%', ha='center', va='bottom')

ax.set_title('Gender Distribution', fontweight='bold')
ax.set_xlabel('Gender')
ax.set_ylabel('Percentage')
plt.tight_layout()
plt.show()

## Regional Distribution

In [ ]:
# Region distribution using SPSS labels
region_data = finished_data.copy()
region_data['region_label'] = apply_spss_labels(region_data, 'region', meta)

region_counts = region_data['region_label'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
region_counts.plot(kind='barh', color='forestgreen', ax=ax)
ax.set_title('Regional Distribution', fontweight='bold')
ax.set_xlabel('Count')
ax.set_ylabel('Region')
plt.tight_layout()
plt.show()

## Housing Ownership Status

In [ ]:
# Ownership status using SPSS labels
ownership_data = finished_data[finished_data['accom1'].notna()].copy()
ownership_data['accom1_label'] = apply_spss_labels(ownership_data, 'accom1', meta)

ownership_counts = ownership_data['accom1_label'].value_counts()
ownership_pct = ownership_counts / ownership_counts.sum() * 100

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Set2(np.linspace(0, 1, len(ownership_pct)))
bars = ax.bar(ownership_pct.index, ownership_pct.values, color=colors)

# Add percentage labels
for bar, pct in zip(bars, ownership_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_title('Housing Ownership Status', fontweight='bold')
ax.set_xlabel('Status')
ax.set_ylabel('Percentage')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Renewable Energy Devices

In [ ]:
# Renewable devices using SPSS variable labels
device_vars = ['accom8_1', 'accom8_2', 'accom8_3', 'accom8_4', 'accom8_5']
device_labels = [get_variable_label(var, meta) for var in device_vars]

# Calculate counts for each device
total = len(finished_data)
device_counts = []
for var in device_vars:
    count = (finished_data[var] == 1).sum()
    device_counts.append(count)

renewable_devices = pd.DataFrame({
    'device_type': device_labels,
    'count': device_counts,
    'percentage': [c / total * 100 for c in device_counts]
})

print(renewable_devices)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
renewable_sorted = renewable_devices.sort_values('percentage', ascending=False)
colors = plt.cm.Set2(np.linspace(0, 1, len(renewable_sorted)))
bars = ax.bar(range(len(renewable_sorted)), renewable_sorted['percentage'], color=colors)

# Add labels
for i, (bar, row) in enumerate(zip(bars, renewable_sorted.itertuples())):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{row.percentage:.1f}%\n(n={row.count})', ha='center', va='bottom', fontsize=9)

ax.set_xticks(range(len(renewable_sorted)))
ax.set_xticklabels(renewable_sorted['device_type'], rotation=45, ha='right')
ax.set_title('Renewable Energy Devices in Accommodations', fontweight='bold')
ax.set_ylabel('Percentage of Households')
plt.tight_layout()
plt.show()

## Heating Sources

In [ ]:
# Heating sources using SPSS labels
heating_data = finished_data[finished_data['heat1_1'].notna()].copy()
heating_data['heat1_1_label'] = apply_spss_labels(heating_data, 'heat1_1', meta)

heating_counts = heating_data['heat1_1_label'].value_counts()
heating_pct = heating_counts / heating_counts.sum() * 100
heating_pct_sorted = heating_pct.sort_values()

print(pd.DataFrame({'source': heating_pct.index, 'pct': heating_pct.values}))

fig, ax = plt.subplots(figsize=(10, 8))
heating_pct_sorted.plot(kind='barh', color='orangered', ax=ax)
ax.set_title('Primary Heating Energy Sources', fontweight='bold')
ax.set_xlabel('Percentage')
ax.set_ylabel('Energy Source')
plt.tight_layout()
plt.show()

## Heating Temperature

In [ ]:
# Heating temperature distribution
temp_data = finished_data[finished_data['heat7'] > 0]['heat7']
mean_temp = temp_data.mean()

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(temp_data, bins=30, color='coral', edgecolor='black', alpha=0.7)
ax.axvline(mean_temp, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_temp:.1f}°C')
ax.set_title('Distribution of Living Room Heating Temperature', fontweight='bold')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## Electricity Consumption Analysis

In [ ]:
# Electricity statistics
elec_data = finished_data[
    (finished_data['elec8_1'] > 0) & 
    (finished_data['elec8_1'] < 50000)
].copy()

elec_stats = {
    'mean_kwh': elec_data['elec8_1'].mean(),
    'median_kwh': elec_data['elec8_1'].median(),
    'sd_kwh': elec_data['elec8_1'].std(),
    'mean_cost': elec_data['elec7_1'].mean(),
    'median_cost': elec_data['elec7_1'].median()
}

print("Electricity Statistics:")
for key, value in elec_stats.items():
    print(f"  {key}: {value:.2f}")

# Electricity cost distribution
cost_data = finished_data[
    (finished_data['elec7_1'] > 0) & 
    (finished_data['elec7_1'] < 5000)
]['elec7_1']

median_cost = cost_data.median()

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(cost_data, bins=50, color='gold', edgecolor='black', alpha=0.7)
ax.axvline(median_cost, color='red', linestyle='--', linewidth=2, 
           label=f'Median: {median_cost:.0f} CHF')
ax.set_title('Distribution of Annual Electricity Costs', fontweight='bold')
ax.set_xlabel('Annual Cost (CHF)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## Car Ownership

In [ ]:
# Car ownership
car_data = finished_data[finished_data['mob2_1'].notna()].copy()
car_counts = car_data['mob2_1'].value_counts().sort_index()
car_pct = car_counts / car_counts.sum() * 100

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(car_pct.index.astype(int), car_pct.values, color='navy')

# Add percentage labels
for bar, pct in zip(bars, car_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', va='bottom')

ax.set_title('Number of Cars per Household', fontweight='bold')
ax.set_xlabel('Number of Cars')
ax.set_ylabel('Percentage')
plt.tight_layout()
plt.show()

## Electric Vehicle Adoption

In [ ]:
# Electric vehicle adoption (mob3_3 == 8 means electric)
car_owners = finished_data[finished_data['mob2_1'] > 0].copy()

total_car_owners = len(car_owners)
ev_owners = ((car_owners['mob3_3'] == 8) | (car_owners['mob2_e'] == 1)).sum()
ev_percentage = ev_owners / total_car_owners * 100

print(f"EV Adoption:")
print(f"  Total car owners: {total_car_owners}")
print(f"  EV owners: {ev_owners}")
print(f"  EV percentage: {ev_percentage:.2f}%")

# Public transport usage
pt_data = {
    'GA Card': (finished_data['mob1_1'] > 0).sum(),
    'Half-fare': (finished_data['mob1_2'] > 0).sum(),
    'Regional': (finished_data['mob1_3'] > 0).sum()
}

total = len(finished_data)
pt_tickets = pd.DataFrame({
    'ticket_type': list(pt_data.keys()),
    'count': list(pt_data.values()),
    'percentage': [c / total * 100 for c in pt_data.values()]
})

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(pt_tickets['ticket_type'], pt_tickets['percentage'], color='purple')

# Add percentage labels
for bar, pct in zip(bars, pt_tickets['percentage']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', va='bottom')

ax.set_title('Public Transport Ticket Ownership', fontweight='bold')
ax.set_xlabel('Ticket Type')
ax.set_ylabel('Percentage of Households')
plt.tight_layout()
plt.show()

## Air Travel Frequency

In [ ]:
# Airplane travel frequency
air_data = finished_data.copy()
air_data['total_flights'] = (
    air_data['mob13_1'].fillna(0) + 
    air_data['mob13_2'].fillna(0) + 
    air_data['mob13_3'].fillna(0) + 
    air_data['mob13_4'].fillna(0)
)
air_data = air_data[air_data['total_flights'].notna()]

fig, ax = plt.subplots(figsize=(12, 6))
flight_counts = air_data['total_flights'].value_counts().sort_index()
# Limit to reasonable range for visualization
flight_counts_limited = flight_counts[flight_counts.index <= 20]
flight_counts_limited.plot(kind='bar', color='skyblue', ax=ax)
ax.set_title('Air Travel Frequency (Last 12 Months)', fontweight='bold')
ax.set_xlabel('Number of Flights')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Climate Change Concerns

In [ ]:
# Climate change concerns using SPSS labels
climate_data = finished_data[
    (finished_data['envpsy3'] >= 0) & 
    (finished_data['envpsy3'].notna())
].copy()

climate_data['envpsy3_label'] = apply_spss_labels(climate_data, 'envpsy3', meta)

# If no labels, use manual mapping
if climate_data['envpsy3_label'].equals(climate_data['envpsy3'].astype(str)):
    worry_labels = {
        1: 'Not at all worried',
        2: 'Slightly worried',
        3: 'Moderately worried',
        4: 'Very worried',
        5: 'Extremely worried'
    }
    climate_data['envpsy3_label'] = climate_data['envpsy3'].map(worry_labels)

climate_counts = climate_data['envpsy3_label'].value_counts()
climate_pct = climate_counts / climate_counts.sum() * 100

# Order the worry levels
worry_order = ['Not at all worried', 'Slightly worried', 'Moderately worried', 
               'Very worried', 'Extremely worried']
climate_pct_ordered = climate_pct.reindex([w for w in worry_order if w in climate_pct.index])

print(pd.DataFrame({'level': climate_pct.index, 'pct': climate_pct.values}))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(climate_pct_ordered.index, climate_pct_ordered.values, color='darkred')

# Add percentage labels
for bar, pct in zip(bars, climate_pct_ordered.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{pct:.1f}%', ha='center', va='bottom')

ax.set_title('Concern About Global Warming', fontweight='bold')
ax.set_xlabel('Worry Level')
ax.set_ylabel('Percentage')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Intention to Reduce Consumption

In [ ]:
# Planned reduction in energy consumption (psy8)
reduction_vars = {
    'electricity': 'psy8_1',
    'heating': 'psy8_2',
    'car_usage': 'psy8_3',
    'carbon_footprint': 'psy8_4',
    'flights': 'psy8_5'
}

reduction_means = {}
for name, var in reduction_vars.items():
    if var in finished_data.columns:
        reduction_means[name] = finished_data[var].mean()

reduction_plans = pd.DataFrame({
    'category': list(reduction_means.keys()),
    'mean_score': list(reduction_means.values())
}).sort_values('mean_score')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(reduction_plans['category'], reduction_plans['mean_score'], color='seagreen')
ax.axvline(3, color='red', linestyle='--', linewidth=1, label='Neutral (3)')
ax.set_title('Intention to Reduce Consumption (Next 12 Months)', fontweight='bold')
ax.set_xlabel('Mean Score\n(1=Very unlikely, 3=Neither likely nor unlikely, 5=Very likely)')
ax.set_ylabel('Category')
ax.legend()
plt.tight_layout()
plt.show()